In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251024_142407"  # "20251028_140930" # "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive

# one regressor
"""--------------------------------------------"""

## init

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
encoder.view_peths()

In [ ]:
encoder_mf.view_fits()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## weight correlation

In [ ]:
from core.data import tv_vals
from core.viz import plot_kdes

weight_diff = {}
for regr in encoder.tv_keys:
    if regr != "response_prev":
        regr_ = f"{regr}_{tv_vals[regr][0]}"
        weight_diff[regr_] = (
            encoder_mb.encoder_weights[:, encoder.dm_idxs[regr_]]
            - encoder_mf.encoder_weights[:, encoder.dm_idxs[regr_]]
        )

plot_kdes(weight_diff)

## pca on DMS/DLS robs

In [ ]:
from sklearn.decomposition import PCA

pca_dls = PCA().fit(encoder.robs[:, encoder.reg_idxs["DLS"]])
pca_dms = PCA().fit(encoder.robs[:, encoder.reg_idxs["DMS"]])
cum_var_dls = np.array([sum(pca_dls.explained_variance_ratio_[:n]) for n in range(100)])
cum_var_dms = np.array([sum(pca_dms.explained_variance_ratio_[:n]) for n in range(100)])

plt.figure(tight_layout=True)
plt.plot(cum_var_dls, label="dls")
plt.plot(cum_var_dms, label="dms")
plt.legend()
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

## pca based on weights

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(encoder.encoder_weights)
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
cum_var = np.array(
    [
        sum(pca.explained_variance_ratio_[:n])
        for n in range(encoder.num_tents + encoder.num_tv)
    ]
)

plt.figure(tight_layout=True)
plt.plot(cum_var)
plt.axhline(y=0.9, color="#666666", linestyle="--")
plt.xlabel("n. components")
plt.ylabel("p(explained variance)")
plt.show()

In [ ]:
n = np.where(cum_var >= 0.9)[0][0]
pca = PCA(n_components=n).fit(encoder.encoder_weights)
weights_lowd = pca.transform(encoder.encoder_weights)

In [ ]:
weights_lowd[:, :3]

In [ ]:
plt.figure()
plt.scatter(weights_lowd[:, 0], weights_lowd[:, 1], alpha=0.5, s=0.5)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection="3d")

ax.scatter(xs=weights_lowd[:, 0], ys=weights_lowd[:, 1], zs=weights_lowd[:, 2], s=0.3)
plt.show()